In [9]:

import pandas as pd
import os

# Universal file loader function
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        df = pd.read_excel(file_path, engine='openpyxl')
    elif ext == '.json':
        df = pd.read_json(file_path)
    elif ext == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    print(f" Loaded file: {file_path}")
    print(f"Shape: {df.shape}")
    print("\nDynamic Attribute Summary:")
    for col in df.columns:
        dtype = df[col].dtype
        sample_values = df[col].dropna().head(3).tolist()
        print(f"Column: {col} | Type: {dtype} | Sample: {sample_values}")

    return df

# Example usage
file_path = r"unified_claims_model_dataset.csv"  # Replace with your file path
df = load_file(file_path)  # df now holds your dataset


 Loaded file: unified_claims_model_dataset.csv
Shape: (100000, 27)

Dynamic Attribute Summary:
Column: patient_id | Type: int64 | Sample: [68642, 67124, 63743]
Column: age | Type: int64 | Sample: [4, 13, 12]
Column: sex | Type: object | Sample: ['F', 'M', 'M']
Column: region | Type: object | Sample: ['Northeast', 'Midwest', 'Midwest']
Column: provider_id | Type: int64 | Sample: [10907, 2495, 4233]
Column: npi_number | Type: int64 | Sample: [7008341820, 7478114055, 3950514574]
Column: speciality | Type: object | Sample: ['Dermatology', 'Radiology', 'Pathology']
Column: provider_organization_name | Type: object | Sample: ['Medical Group 817', 'P.C. 437', 'Medical Group 940']
Column: provider_lastname | Type: object | Sample: ['Miller', 'Williams', 'Brown']
Column: provider_firstname | Type: object | Sample: ['Mary', 'Robert', 'Patricia']
Column: provider_middlename | Type: object | Sample: ['A', 'D', 'D']
Column: employer_identification_number | Type: object | Sample: ['57-9602367', '33-

In [10]:
df.isnull().sum()

patient_id                            0
age                                   0
sex                                   0
region                                0
provider_id                           0
npi_number                            0
speciality                            0
provider_organization_name            0
provider_lastname                     0
provider_firstname                    0
provider_middlename               59993
employer_identification_number        0
provider_address                      0
provider_city_name                    0
provider_state_name                   0
provider_postalcode                   0
procedure_code                        0
procedure_description                 0
category                              0
date_of_service                       0
pre_adjudication_date                 0
adjudication_date                     0
payment_date                          0
adjudication_status                   0
billed_amount                         0


In [11]:
import hashlib
# Helper function for hashing IDs
def hash_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()


In [12]:

# Create dim_patient from original dataset
dim_patient = df[['patient_id', 'age', 'sex', 'region']].drop_duplicates().reset_index(drop=True)

# Add surrogate key
dim_patient['patient_sk'] = range(1, len(dim_patient) + 1)

# Hash patient_id for privacy
def hash_id(x):
    return hash(str(x))

dim_patient['patient_id_hashed'] = dim_patient['patient_id'].apply(hash_id)

# Select final columns
dim_patient = dim_patient[['patient_sk', 'patient_id_hashed', 'age', 'sex', 'region']]
dim_patient.to_csv('dim_patient.csv', index=False)
# Preview
print(dim_patient.head())


   patient_sk    patient_id_hashed  age sex     region
0           1  5815388412655108041    4   F  Northeast
1           2 -1147195692216404442   13   M    Midwest
2           3 -7545747547411593430   12   M    Midwest
3           4  7334834523305071874    1   M  Northeast
4           5 -2194824862195193771   75   M  Northeast


In [14]:
import pandas as pd

# Load data
df = pd.read_csv("unified_claims_model_dataset.csv")

# Fix column names based on your dataset
dim_provider = df[['provider_id', 'speciality', 'npi_number']].drop_duplicates().reset_index(drop=True)

# Create surrogate key
dim_provider['provider_sk'] = range(1, len(dim_provider) + 1)

# Handle missing values
dim_provider['provider_id'] = dim_provider['provider_id'].fillna(-1)
dim_provider['npi_number'] = dim_provider['npi_number'].fillna(-1)
dim_provider['speciality'] = dim_provider['speciality'].fillna("Unknown")

# Reorder columns
dim_provider = dim_provider[['provider_sk', 'provider_id', 'speciality', 'npi_number']]

# Add default Unknown provider if needed
default_provider_sk = 0
if -1 in dim_provider['provider_id'].values:
    unknown_row = pd.DataFrame(
        [[default_provider_sk, -1, 'Unknown', -1]],
        columns=['provider_sk', 'provider_id', 'speciality', 'npi_number']
    )
    dim_provider = pd.concat([unknown_row, dim_provider], ignore_index=True)

# Save
dim_provider.to_csv('dim_provider.csv', index=False)

dim_provider.head()



,provider_sk,provider_id,speciality,npi_number
0,1,10907,Dermatology,7008341820
1,2,2495,Radiology,7478114055
2,3,4233,Pathology,3950514574
3,4,7840,Orthopedics,8485370455
4,5,7660,Orthopedics,9146332360


In [16]:
print(cpt_df.columns)
print(icd_df.columns)


Index(['procedure_code', 'cpt_description'], dtype='object')
Index(['Diagnosis', 'procedure_code', 'icd_description', 'Similarity Score',
       'Justification', 'Alternative Suggestions', 'Needs Review'],
      dtype='object')


In [17]:
import pandas as pd

# Load datasets
claims_df = pd.read_csv("unified_claims_model_dataset.csv")
cpt_df = pd.read_csv("cpt4.csv")
icd_df = pd.read_csv("icd10_mapped_output.csv")

# --- STEP 0: Standardize procedure_code as STRING everywhere ---
claims_df['procedure_code'] = claims_df['procedure_code'].astype(str)
cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'] = \
    cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'].astype(str)
icd_df['ICD-10 Code'] = icd_df['ICD-10 Code'].astype(str)

# --- Rename columns ---
cpt_df = cpt_df.rename(columns={
    'com.medigy.persist.reference.type.clincial.CPT.code': 'procedure_code',
    'label': 'cpt_description'
})
icd_df = icd_df.rename(columns={
    'ICD-10 Code': 'procedure_code',
    'ICD Description': 'icd_description'
})

# --- STEP 1: Extract procedure data ---
dim_procedure = claims_df[['procedure_code', 'procedure_description', 'category']].copy()

# --- STEP 2: Merge CPT descriptions ---
dim_procedure = dim_procedure.merge(
    cpt_df[['procedure_code', 'cpt_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 3: Merge ICD descriptions ---
dim_procedure = dim_procedure.merge(
    icd_df[['procedure_code', 'icd_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 4: Choose best description ---
dim_procedure['final_description'] = (
    dim_procedure['procedure_description']
        .fillna(dim_procedure['cpt_description'])
        .fillna(dim_procedure['icd_description'])
        .fillna("Unknown Procedure")
)

# --- STEP 5: Fill missing category ---
dim_procedure['category'] = dim_procedure['category'].fillna("Uncategorized")

# --- STEP 6: Remove duplicates ---
dim_procedure = dim_procedure[['procedure_code', 'final_description', 'category']].drop_duplicates()

# --- STEP 7: Create surrogate key ---
dim_procedure['proc_sk'] = range(1, len(dim_procedure) + 1)

# --- Final order ---
dim_procedure = dim_procedure[['proc_sk', 'procedure_code', 'final_description', 'category']]

# Save output
dim_procedure.to_csv("dim_procedure.csv", index=False)

dim_procedure.head()


,proc_sk,procedure_code,final_description,category
0,1,71046,"Radiologic examination, chest; 2 views, fronta...",Imaging
1,2,71045,"Radiologic examination, chest; single view, fr...",Imaging
2,3,82947,"Glucose; quantitative, blood (except reagent s...",Labs
3,4,81002,"Urinalysis, by dipstick or tablet reagent for ...",Labs
4,5,85025,Complete (CBC) and automated differential WBC ...,Labs


In [18]:

# # Map patient_sk from dim_patient
# patient_map = dict(zip(df['patient_id'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

# # Map provider_sk from dim_provider
# provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
# fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# # Map proc_sk from dim_procedure
# proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
# fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# # Add claim_sk
# fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)
df.head()

,patient_id,age,sex,region,provider_id,npi_number,speciality,provider_organization_name,provider_lastname,provider_firstname,...,procedure_description,category,date_of_service,pre_adjudication_date,adjudication_date,payment_date,adjudication_status,billed_amount,paid_amount,diagnosis_code
0,68642,4,F,Northeast,10907,7008341820,Dermatology,Medical Group 817,Miller,Mary,...,"Radiologic examination, chest; 2 views, fronta...",Imaging,2023-02-07T00:00:00,2023-02-11T00:00:00,2023-02-23T00:00:00,2023-03-23T00:00:00,75%,3090.28,2317.71,Z00.00
1,67124,13,M,Midwest,2495,7478114055,Radiology,P.C. 437,Williams,Robert,...,"Radiologic examination, chest; single view, fr...",Imaging,2022-03-09T00:00:00,2022-03-15T00:00:00,2022-03-27T00:00:00,2022-04-19T00:00:00,100%,1464.20,1464.20,K21.9
2,63743,12,M,Midwest,4233,3950514574,Pathology,Medical Group 940,Brown,Patricia,...,"Glucose; quantitative, blood (except reagent s...",Labs,2022-10-18T00:00:00,2022-10-25T00:00:00,2022-10-31T00:00:00,2022-11-01T00:00:00,100%,2094.59,2094.59,M54.5
3,46652,1,M,Northeast,7840,8485370455,Orthopedics,Health Center 902,Williams,Jane,...,"Urinalysis, by dipstick or tablet reagent for ...",Labs,2022-06-20T00:00:00,2022-06-30T00:00:00,2022-07-07T00:00:00,2022-07-13T00:00:00,50%,1506.43,753.22,J02.9
4,63742,75,M,Northeast,7660,9146332360,Orthopedics,Clinic 966,Smith,Linda,...,Complete (CBC) and automated differential WBC ...,Labs,2022-05-01T00:00:00,2022-05-10T00:00:00,2022-05-16T00:00:00,2022-05-31T00:00:00,75%,527.26,395.44,E11.9


In [19]:

import pandas as pd

# Assuming df is your main claims dataset
# Load updated dataset with age categories
df = pd.read_csv("unified_claims_model_dataset.csv")

# --- Create fact_claim ---
fact_claim = df[['patient_id', 'provider_id', 'procedure_code', 'date_of_service',
                 'billed_amount', 'paid_amount', 'adjudication_status']].copy()

# Add surrogate key for fact table
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# # Map patient_sk from dim_patient
# patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

fact_claim['patient_id_hashed'] = fact_claim['patient_id'].apply(hash_id)
patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
fact_claim['patient_sk'] = fact_claim['patient_id_hashed'].map(patient_map)


# Map provider_sk from dim_provider
provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# Map proc_sk from dim_procedure
proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# Drop original IDs (optional)
fact_claim = fact_claim[['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk',
                         'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status']]

# Validate
print("✅ fact_claim table created successfully!")
print(fact_claim.head())

# Save to CSV
fact_claim.to_csv('fact_claim.csv', index=False)


✅ fact_claim table created successfully!
   claim_sk  patient_sk  provider_sk  proc_sk      date_of_service  \
0         1       48506        91433      NaN  2023-02-07T00:00:00   
1         2           2        92273      NaN  2022-03-09T00:00:00   
2         3           3        96364      NaN  2022-10-18T00:00:00   
3         4           4        96582      NaN  2022-06-20T00:00:00   
4         5           5        89588      NaN  2022-05-01T00:00:00   

   billed_amount  paid_amount adjudication_status  
0        3090.28      2317.71                 75%  
1        1464.20      1464.20                100%  
2        2094.59      2094.59                100%  
3        1506.43       753.22                 50%  
4         527.26       395.44                 75%  


In [20]:

# Add surrogate key first
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# Now run validation
validation_results = []

# 1. Uniqueness Checks
checks = {
    'dim_patient.patient_sk': dim_patient['patient_sk'].is_unique,
    'dim_provider.provider_sk': dim_provider['provider_sk'].is_unique,
    'dim_procedure.proc_sk': dim_procedure['proc_sk'].is_unique,
    'fact_claim.claim_sk': fact_claim['claim_sk'].is_unique
}
for name, result in checks.items():
    validation_results.append({'Check': f'Unique {name}', 'Status': 'PASS' if result else 'FAIL'})

# 2. Null Checks in Critical Columns
critical_fact_cols = ['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk', 'date_of_service']
for col in critical_fact_cols:
    null_count = fact_claim[col].isnull().sum()
    validation_results.append({'Check': f'Nulls in fact_claim.{col}', 'Status': 'PASS' if null_count == 0 else f'FAIL ({null_count} nulls)'})

# 3. Referential Integrity Checks
invalid_patient_refs = fact_claim[~fact_claim['patient_sk'].isin(dim_patient['patient_sk'])]
validation_results.append({'Check': 'Referential integrity patient_sk', 'Status': 'PASS' if invalid_patient_refs.empty else f'FAIL ({len(invalid_patient_refs)} invalid)'})

invalid_provider_refs = fact_claim[~fact_claim['provider_sk'].isin(dim_provider['provider_sk'])]
validation_results.append({'Check': 'Referential integrity provider_sk', 'Status': 'PASS' if invalid_provider_refs.empty else f'FAIL ({len(invalid_provider_refs)} invalid)'})

invalid_proc_refs = fact_claim[~fact_claim['proc_sk'].isin(dim_procedure['proc_sk'])]
validation_results.append({'Check': 'Referential integrity proc_sk', 'Status': 'PASS' if invalid_proc_refs.empty else f'FAIL ({len(invalid_proc_refs)} invalid)'})

# Export validation report
report_df = pd.DataFrame(validation_results)
report_df.to_csv('validation_report.csv', index=False)

print("✅ Validation completed. Report saved as validation_report.csv")
print(report_df)


✅ Validation completed. Report saved as validation_report.csv
                                  Check                 Status
0         Unique dim_patient.patient_sk                   PASS
1       Unique dim_provider.provider_sk                   PASS
2          Unique dim_procedure.proc_sk                   PASS
3            Unique fact_claim.claim_sk                   PASS
4          Nulls in fact_claim.claim_sk                   PASS
5        Nulls in fact_claim.patient_sk                   PASS
6       Nulls in fact_claim.provider_sk                   PASS
7           Nulls in fact_claim.proc_sk    FAIL (100000 nulls)
8   Nulls in fact_claim.date_of_service                   PASS
9      Referential integrity patient_sk                   PASS
10    Referential integrity provider_sk                   PASS
11        Referential integrity proc_sk  FAIL (100000 invalid)


In [21]:

import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load your DataFrames into SQLite
dim_patient.to_sql('dim_patient', conn, index=False)
dim_provider.to_sql('dim_provider', conn, index=False)
dim_procedure.to_sql('dim_procedure', conn, index=False)
fact_claim.to_sql('fact_claim', conn, index=False)

# 1. Top Providers by Paid Amount

query1 = """
SELECT p.provider_id, p.speciality, SUM(f.paid_amount) AS total_paid
FROM fact_claim f
JOIN dim_provider p ON f.provider_sk = p.provider_sk
GROUP BY p.provider_id, p.speciality
ORDER BY total_paid DESC
LIMIT 10;
"""
result1 = pd.read_sql_query(query1, conn)
print("\nTop Providers by Paid Amount:")
print(result1)






Top Providers by Paid Amount:
   provider_id        speciality  total_paid
0         6383       Orthopedics    61094.40
1         9573        Cardiology    51582.50
2         9660       Orthopedics    50832.41
3         5439       Orthopedics    50502.19
4         6738       Orthopedics    49206.20
5         1027         Radiology    49190.98
6        10949       Dermatology    47616.25
7         3738        Pediatrics    47454.06
8         3384  General Practice    47084.24
9         7883       Orthopedics    46319.16


In [5]:

import os
import shutil
from datetime import datetime

source_file = 'fact_claim.csv'
landing_zone = 'landing/fact_claim/'
staging_zone = 'staging/fact_claim/'

# Create directories if they don't exist
os.makedirs(landing_zone, exist_ok=True)
os.makedirs(staging_zone, exist_ok=True)

# Step 1: Move file to landing zone
shutil.copy(source_file, landing_zone)

# Step 2: Log metadata
log_file = 'ingestion_log.txt'
with open(log_file, 'a') as log:
    log.write(f"{source_file}, {datetime.now()}, {os.path.getsize(source_file)} bytes\n")

# Step 3: Validate and move to staging
if os.path.exists(os.path.join(landing_zone, source_file)):
    shutil.move(os.path.join(landing_zone, source_file), staging_zone)
    print("File moved to staging successfully!")


Error: Destination path 'staging/fact_claim/fact_claim.csv' already exists

In [3]:
print(os.listdir('staging/fact_claim'))

['fact_claim.csv']


In [1]:

import pandas as pd

# Load the unified dataset
unified_df = pd.read_csv('unified_claims_model_dataset.csv')

# Select relevant columns for ingestion layer fact_claim
fact_claim_ingestion = unified_df[[
    'patient_id', 'provider_id', 'region', 'procedure_code', 'procedure_description',
    'date_of_service', 'pre_adjudication_date', 'adjudication_date', 'payment_date',
    'adjudication_status', 'billed_amount', 'paid_amount'
]].copy()

# Rename columns to match fact_claim structure
fact_claim_ingestion.rename(columns={
    'patient_id': 'patient_sk',
    'provider_id': 'provider_sk'
}, inplace=True)

# Add claim_sk as a sequential ID
fact_claim_ingestion.insert(0, 'claim_sk', range(1, len(fact_claim_ingestion) + 1))

# Save the enriched ingestion layer table
fact_claim_ingestion.to_csv('fact_claim_ingestion.csv', index=False)

print("Ingestion layer fact_claim table created successfully with new columns:")
print(fact_claim_ingestion.head())
print(f"Total rows: {len(fact_claim_ingestion)}")


Ingestion layer fact_claim table created successfully with new columns:
   claim_sk  patient_sk  provider_sk     region  procedure_code  \
0         1       68642        10907  Northeast           71046   
1         2       67124         2495    Midwest           71045   
2         3       63743         4233    Midwest           82947   
3         4       46652         7840  Northeast           81002   
4         5       63742         7660  Northeast           85025   

                               procedure_description      date_of_service  \
0  Radiologic examination, chest; 2 views, fronta...  2023-02-07T00:00:00   
1  Radiologic examination, chest; single view, fr...  2022-03-09T00:00:00   
2  Glucose; quantitative, blood (except reagent s...  2022-10-18T00:00:00   
3  Urinalysis, by dipstick or tablet reagent for ...  2022-06-20T00:00:00   
4  Complete (CBC) and automated differential WBC ...  2022-05-01T00:00:00   

  pre_adjudication_date    adjudication_date         payment_d